# Financial News Sentiment Analysis — Results & Analysis

This notebook loads all persisted results and generates figures/tables for the report.

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from IPython.display import display

# Load evaluation results
eval_results = json.loads(Path("../results/metrics/evaluation_results.json").read_text())
data_stats = json.loads(Path("../results/metrics/data_stats.json").read_text())
market_trend = json.loads(Path("../results/metrics/market_trend.json").read_text())

per_window = pd.DataFrame(eval_results["per_window"])
aggregated = pd.DataFrame(eval_results["aggregated"])

print("Data Stats:")
print(json.dumps(data_stats, indent=2))

## 1. Data Description

In [ ]:
# Dataset overview
print(f"Total article-ticker pairs: {data_stats['total_article_ticker_pairs']}")
print(f"Unique articles: {data_stats['unique_articles']}")
print(f"Unique tickers: {data_stats['unique_tickers']}")
print(f"Date range: {data_stats['date_range']}")

# Window sizes
window_df = pd.DataFrame(data_stats["windows"]).T
display(window_df)

# Horizon availability
horizon_df = pd.DataFrame([data_stats["horizon_availability"]])
display(horizon_df)

## 2. Fine-Tuned Model Evaluation: R² and MSE

In [ ]:
ft = aggregated[aggregated["model_type"] == "finetuned"].copy()

for metric in ["r2_mean", "mse_mean"]:
    for approach in ["separate", "single"]:
        subset = ft[ft["approach"] == approach]
        if subset.empty:
            continue
        pivot = subset.pivot(index="model", columns="horizon", values=metric)
        fig, ax = plt.subplots(figsize=(10, 4))
        sns.heatmap(pivot, annot=True, fmt=".4f", cmap="RdYlGn" if "r2" in metric else "RdYlGn_r", ax=ax)
        ax.set_title(f"{metric} — {approach} approach")
        fig.tight_layout()
        fig.savefig(f"../results/figures/heatmap_{metric}_{approach}.png", dpi=150)
        plt.show()

## 3. Training Loss Curves

In [ ]:
import glob

log_files = sorted(glob.glob("../results/training_logs/*_log.json"))

# Plot one example per encoder (best config, r_1d, window_1)
for encoder in ["bert", "finbert", "roberta"]:
    log_path = f"../results/training_logs/{encoder}_separate_1d_window_1_log.json"
    if not Path(log_path).exists():
        continue
    log = json.loads(Path(log_path).read_text())
    logs = pd.DataFrame(log["logs"])

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(logs["epoch"], logs["train_loss"], label="Train")
    ax.plot(logs["epoch"], logs["val_loss"], label="Validation")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("MSE Loss")
    ax.set_title(f"{encoder} — r_1d — Window 1")
    ax.legend()
    fig.tight_layout()
    fig.savefig(f"../results/figures/loss_curve_{encoder}_1d_w1.png", dpi=150)
    plt.show()

## 4. Binary Prediction: Accuracy & F1

In [ ]:
# Compare fixed vs learned threshold
binary_cols = ["model", "model_type", "approach", "horizon",
               "fixed_accuracy_mean", "fixed_f1_mean",
               "learned_accuracy_mean", "learned_f1_mean"]
binary_df = aggregated[[c for c in binary_cols if c in aggregated.columns]].copy()
display(binary_df.round(4))

# Bar chart: F1 by model and horizon
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, (threshold_type, col) in zip(axes, [("Fixed (0)", "fixed_f1_mean"), ("Learned", "learned_f1_mean")]):
    if col not in aggregated.columns:
        continue
    pivot = aggregated.pivot_table(index="model", columns="horizon", values=col, aggfunc="mean")
    pivot.plot(kind="bar", ax=ax)
    ax.set_title(f"Macro F1 — {threshold_type} Threshold")
    ax.set_ylabel("F1 Score")
    ax.legend(title="Horizon", bbox_to_anchor=(1.05, 1))
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45)
fig.tight_layout()
fig.savefig("../results/figures/binary_f1_comparison.png", dpi=150)
plt.show()

## 5. Separate vs Single Model Comparison

In [ ]:
ft_only = aggregated[aggregated["model_type"] == "finetuned"]
comparison = ft_only.pivot_table(
    index=["model", "horizon"],
    columns="approach",
    values=["r2_mean", "mse_mean"],
    aggfunc="mean"
)
display(comparison.round(4))

# Difference plot
for metric in ["r2_mean", "mse_mean"]:
    if metric not in ft_only.columns:
        continue
    sep = ft_only[ft_only["approach"] == "separate"].set_index(["model", "horizon"])[metric]
    sin = ft_only[ft_only["approach"] == "single"].set_index(["model", "horizon"])[metric]
    diff = (sep - sin).reset_index()
    diff.columns = ["model", "horizon", "difference"]
    pivot = diff.pivot(index="model", columns="horizon", values="difference")
    fig, ax = plt.subplots(figsize=(10, 4))
    sns.heatmap(pivot, annot=True, fmt=".4f", center=0, cmap="RdBu", ax=ax)
    ax.set_title(f"Separate - Single ({metric})")
    fig.tight_layout()
    fig.savefig(f"../results/figures/approach_diff_{metric}.png", dpi=150)
    plt.show()

## 6. Sentiment & Market Trend Analysis

In [ ]:
trend_df = pd.DataFrame(market_trend)
display(trend_df.round(4))

# Display saved scatter plots
from IPython.display import Image
import glob

scatter_files = sorted(glob.glob("../results/figures/scatter_*.png"))
for f in scatter_files:
    print(f"\n{Path(f).stem}")
    display(Image(filename=f, width=600))

# Time series overlay
overlay_path = "../results/figures/time_series_overlay.png"
if Path(overlay_path).exists():
    display(Image(filename=overlay_path, width=900))